# 03 — Slides

How the system handles lecture slide decks, measured on the fifteen AI lecture decks in `data/projects/AI` with the slide ground truth (`data/eval/slide_ground_truth.json`). Three questions:

1. Should a slide be packed together with its neighbours or embedded alone?
2. Does the system answer questions about the student's own decks?
3. What does reading a slide as a picture buy, and what does it cost?

**Running it.** Every section below is the evaluation's own code. With `RUN = False`
(the default) nothing is recomputed: the results saved in `data/eval/` are loaded
and shown. Set `RUN = True` in the first code cell to measure again, which
overwrites those files. Chunking and answering take a few minutes each; the picture-reading run about an hour, most of it Qwen2-VL on picture-only slides.

In [1]:
import os
import sys

sys.path.insert(0, os.path.abspath("") if os.path.basename(os.path.abspath("")) == "notebooks"
                else os.path.join(os.path.abspath(""), "notebooks"))
from eval_common import repo_root  # noqa: E402

REPO_ROOT = repo_root()
sys.path.insert(0, str(REPO_ROOT))
sys.path.insert(0, str(REPO_ROOT / "notebooks"))

# The application's settings (backend/service.py). They are read when the
# pipeline modules are imported, so they are set before anything else.
os.environ.setdefault("SA_EMBEDDER", "bge-small")

# False: show the results saved in data/eval. True: run the evaluation again
# and overwrite them (the run time is given at the top of the notebook).
RUN = False

In [2]:
import json

import pandas as pd

EVAL_DIR = REPO_ROOT / "data" / "eval"


def saved(name):
    """A results file from data/eval."""
    return json.loads((EVAL_DIR / name).read_text(encoding="utf-8"))

## Packing slides or embedding them one by one

Tests the decision that slides are *packed* rather than split.

Prose pages hold more text than a chunk, so the chunker cuts them up. A slide
holds far less, and the first version of this system embedded each slide on
its own: chunks of a median 19 tokens, most of them a title and three bullet
points, which is too little context for an embedding to mean much.
preprocessor._pack_slides now packs consecutive slides up to the same
220-token limit and starts a new chunk when the slide title changes.

Four ways of chunking the same fifteen decks are compared on the slide ground
truth (build_slide_ground_truth.py), whose questions were written from single
slides before any chunker ran, so no strategy is favoured by construction:

```text
  packed     — the system's rule: pack until full or until the title changes
  per_slide  — one chunk per slide, the original behaviour
  window     — treat a deck like prose: fixed 220-token windows, 40 overlap
  sentence   — treat a deck like prose: sentence-aware windows
```

A question is a hit at k when one of the top k chunks comes from the labelled
file and its page range covers the labelled slide. Both retrieval modes the
app can run are reported, because a packing rule that only looks good with
BM25 alongside it would be a different claim.

Only text_layer questions are used; the picture questions are what
eval_figure_reading.py measures. Pictures are still read when building the
index, as the app reads them, so every mode sees the same text.

In [3]:
# The script's command-line options, as it would have read them.
sys.argv = ['notebook']

In [4]:
import json
import os
import statistics
import sys
import time
from pathlib import Path

ROOT = REPO_ROOT
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

import faiss  # noqa: E402
import numpy as np  # noqa: E402

from backend.pipeline import sparse as sparse_module  # noqa: E402
from backend.pipeline.chunk import Chunk  # noqa: E402
from backend.pipeline.embedder import _get_model, embed, model_key  # noqa: E402
from backend.pipeline.loader import load_pdf  # noqa: E402
from backend.pipeline.preprocessor import preprocess  # noqa: E402
from backend.pipeline.retriever import DENSE_WEIGHT, retrieve  # noqa: E402

DECK_DIR = ROOT / "data" / "projects" / "AI"
EVAL_DIR = ROOT / "data" / "eval"
GROUND_TRUTH_PATH = EVAL_DIR / "slide_ground_truth.json"
OUT_PATH = EVAL_DIR / "slide_chunking.json"

K_VALUES = [1, 3, 5]
MODES = ["packed", "per_slide", "window", "sentence"]

In [5]:
def chunks_for(mode: str, pages_by_deck):
    """Every deck chunked one way, as one list."""
    chunks = []
    for pages in pages_by_deck:
        if mode == "packed":
            chunks.extend(preprocess(pages))          # kind == "slide" -> packed
        elif mode == "per_slide":
            chunks.extend(Chunk(" ".join(p["text"].split()), p["source_file"],
                                p["page"], p.get("section"), kind="slide")
                          for p in pages if p["text"].strip())
        else:
            # Hide the fact that these are slides, so the prose path runs.
            as_prose = [{k: v for k, v in p.items() if k != "kind"} for p in pages]
            chunks.extend(preprocess(as_prose, chunking=mode))
    return chunks

In [6]:
def covers(chunk, source_file, page) -> bool:
    if getattr(chunk, "source_file", None) != source_file:
        return False
    first = getattr(chunk, "page", None)
    if first is None:
        return False
    last = getattr(chunk, "page_end", None) or first
    return first <= page <= last

In [7]:
def slides_in(chunks) -> int:
    """How many distinct slides a set of chunks spans."""
    seen = set()
    for c in chunks:
        first = getattr(c, "page", None)
        if first is None:
            continue
        last = getattr(c, "page_end", None) or first
        seen.update((getattr(c, "source_file", None), p)
                    for p in range(first, last + 1))
    return len(seen)

In [8]:
def evaluate(chunks, questions, keywords, index, hybrid: bool):
    ranks, covered = [], []
    for entry in questions:
        got = retrieve(entry["question"], index, chunks, k=max(K_VALUES),
                       sparse=keywords if hybrid else None,
                       dense_weight=DENSE_WEIGHT)
        ranks.append(next((i for i, c in enumerate(got, 1)
                           if covers(c, entry["source_file"], entry["page"])), None))
        covered.append(slides_in(got[:3]))
    total = len(questions)
    return {
        "recall": {f"recall_at_{k}": round(sum(1 for r in ranks if r and r <= k) / total, 3)
                   for k in K_VALUES},
        "mrr_top5": round(statistics.mean(1 / r if r else 0 for r in ranks), 4),
        # How much of the deck the top 3 chunks put in front of the reader. A
        # packed chunk spans several slides, so it has more chances to cover
        # the labelled one: recall alone would flatter it, and this is the
        # measure that says by how much.
        "slides_covered_at_3": round(statistics.mean(covered), 1),
        "ranks": ranks,
    }

In [9]:
def main():
    # Slide text carries Unicode the Windows console cannot encode.
    try:
        sys.stdout.reconfigure(encoding="utf-8", errors="replace")
    except Exception:
        pass
    truth = json.loads(GROUND_TRUTH_PATH.read_text(encoding="utf-8"))
    questions = [q for q in truth["questions"] if q["source"] == "text_layer"]
    print(f"{len(questions)} text-layer questions over {truth['decks']} decks")
    print(f"embedder: {model_key()}   dense weight: {DENSE_WEIGHT}\n")

    paths = sorted(p for p in DECK_DIR.glob("*.pdf"))
    pages_by_deck = [load_pdf(str(p), figures="auto") for p in paths]
    tokenizer = _get_model().tokenizer

    results = {}
    print(f"{'mode':<12} {'chunks':>7} {'median tok':>11} "
          f"{'R@1':>6} {'R@3':>6} {'R@5':>6} {'MRR':>7} {'slides':>7}   retrieval")
    for mode in MODES:
        t0 = time.time()
        chunks = chunks_for(mode, pages_by_deck)
        lengths = [len(tokenizer.encode(str(c), add_special_tokens=False))
                   for c in chunks]
        embeddings = embed(chunks)
        index = faiss.IndexFlatL2(embeddings.shape[1])
        index.add(np.ascontiguousarray(embeddings, dtype=np.float32))
        keywords = sparse_module.build_index(chunks)

        results[mode] = {
            "chunks": len(chunks),
            "median_tokens": round(statistics.median(lengths), 1),
            "mean_tokens": round(statistics.mean(lengths), 1),
            "build_seconds": round(time.time() - t0, 1),
            "hybrid": evaluate(chunks, questions, keywords, index, True),
            "dense_only": evaluate(chunks, questions, keywords, index, False),
        }
        for label in ("hybrid", "dense_only"):
            r = results[mode][label]
            print(f"{mode if label == 'hybrid' else '':<12} "
                  f"{len(chunks) if label == 'hybrid' else '':>7} "
                  f"{results[mode]['median_tokens'] if label == 'hybrid' else '':>11} "
                  f"{r['recall']['recall_at_1']:>6} {r['recall']['recall_at_3']:>6} "
                  f"{r['recall']['recall_at_5']:>6} {r['mrr_top5']:>7} "
                  f"{r['slides_covered_at_3']:>7}   {label}")

    base = results["per_slide"]["hybrid"]["ranks"]
    packed = results["packed"]["hybrid"]["ranks"]
    gained = [i + 1 for i, (a, b) in enumerate(zip(packed, base))
              if (a and a <= 3) and not (b and b <= 3)]
    lost = [i + 1 for i, (a, b) in enumerate(zip(packed, base))
            if (b and b <= 3) and not (a and a <= 3)]
    print(f"\n  packed vs per_slide at k=3 (hybrid): gained {gained}, lost {lost}")

    OUT_PATH.write_text(json.dumps({
        "note": "Slide chunking strategies on the slide ground truth. A hit at "
                "k means a top-k chunk is from the labelled file and its page "
                "range covers the labelled slide. ranks: first such rank in "
                "the top 5, null if none. Question numbers in gained/lost are "
                "1-based positions in the text_layer question list. "
                "slides_covered_at_3 is how many distinct slides the top 3 "
                "chunks span on average: a packed chunk covers several, so it "
                "has more chances to satisfy the hit rule.",
        "questions": len(questions),
        "decks": len(paths),
        "embedder": model_key(),
        "dense_weight": DENSE_WEIGHT,
        "packed_vs_per_slide_at_3": {"gained": gained, "lost": lost},
        "modes": results,
    }, indent=2) + "\n", encoding="utf-8")
    print(f"\n  -> {OUT_PATH.relative_to(ROOT)}")

In [10]:
if RUN:
    main()
else:
    print('RUN is False: showing the saved results below.')

RUN is False: showing the saved results below.


### Results

In [11]:
sc = saved("slide_chunking.json")
rows = []
for mode, m in sc["modes"].items():
    for retrieval in ("hybrid", "dense_only"):
        r = m[retrieval]
        rows.append({"mode": mode, "retrieval": retrieval, "chunks": m["chunks"],
                     "median tokens": m["median_tokens"], **r["recall"],
                     "MRR@5": r["mrr_top5"], "slides in top 3": r["slides_covered_at_3"]})
print(f"{sc['questions']} questions over {sc['decks']} decks; packed vs per-slide at k=3:",
      sc["packed_vs_per_slide_at_3"])
pd.DataFrame(rows).set_index(["mode", "retrieval"])

38 questions over 15 decks; packed vs per-slide at k=3: {'gained': [9, 20, 25], 'lost': [17]}


chunks  median tokens  recall_at_1  recall_at_3  \
mode      retrieval                                                     
packed    hybrid         228          129.0        0.711        0.816   
          dense_only     228          129.0        0.632        0.737   
per_slide hybrid         672           29.0        0.605        0.763   
          dense_only     672           29.0        0.763        0.868   
window    hybrid         677           29.0        0.605        0.763   
          dense_only     677           29.0        0.763        0.868   
sentence  hybrid         678           29.0        0.605        0.763   
          dense_only     678           29.0        0.763        0.868   

                      recall_at_5   MRR@5  slides in top 3  
mode      retrieval                                         
packed    hybrid            0.842  0.7697              9.4  
          dense_only        0.868  0.7075              9.2  
per_slide hybrid            0.816  0.6873              3.0  
          dense_only        0.895  0.8211              3.0  
window    hybrid            0.816  0.6873              3.0  
          dense_only        0.895  0.8224              3.0  
sentence  hybrid            0.816  0.6873              3.0  
          dense_only        0.895  0.8224              3.0

## Answering questions on the student's own decks

Answers the question a document benchmark cannot: does the system work for the
job it was built for?

Such a benchmark asks a retrieval pipeline to read charts, count marks on a
page and join facts across a fifty-page financial report. It scores 6.2% on the
questions that have an answer, and that is a real result about the generator.
It is not a result about revising from your own lecture slides, which is what
this system is for, and nothing measured so far covers that end to end: the
slide ground truth was used only to score retrieval.

So the same 51 questions over the student's own fifteen lecture decks are run
all the way through, in both styles the application uses:

```text
  short   — the extractive answer, which the quiz compares against
  explain — the paragraph the chat view streams
```

Scored by generation_analysis.judge, the same rule as every other end-to-end
number here.

#### Why the automatic score here is not trustworthy, and what to use instead
Measured: 12/51 short, 15/51 explain. Those numbers should not be quoted.
Thirty-six of the fifty-one reference answers are two words or fewer, because
they were produced by the short style from a single slide: "CGAT", "SLIDING",
"100100", "writing tests". A fragment like that is a fine label for *which
slide* the answer is on, which is what the retrieval evaluation uses it for.
It is a poor reference for whether a paragraph is right, and the judge marks
plainly correct answers wrong on it:

```text
    "How many songs were in the dataset?"  reference "200k",
        answered "approximately 200,000 songs"            marked wrong
    "What are the labels of the nodes?"    reference "A, B, C",
        answered "The nodes are labeled as A, B, and C."  marked wrong
    "What are the Four Ps of creativity?"  reference "Product, Process,
        Press/Environment", answered with all three and Person/Producer as
        well                                             marked wrong
```

There are real failures in there too — SketchRNN answered as "Dall-e", 1987 as
2006 — but the two cannot be separated by this rule, so the script also writes
a blind rating sheet, own_material_rating_sheet.csv, with the answers in a
random order and no indication of how they were scored. Ten minutes of ticking
settles what the automatic rule cannot.

#### What this can and cannot show
The questions were written by this system's own question model from single
slides, so the reference answer is one this system produced with that slide in
hand. Retrieval then has to find that slide again among all fifteen decks, and
the model has to answer from whatever comes back. That makes this a round-trip
consistency measure in the sense of Alberti et al. (2019) — can the pipeline
recover, through retrieval, the answer it would have given with the page in
front of it — and not a measure of absolute correctness. It is biased upward
for exactly that reason and the report must say so.

It is still the closest thing to the real task, on the real material, and it
is the number that says whether the application does its job.

In [12]:
# The script's command-line options, as it would have read them.
sys.argv = ['notebook']

In [13]:
import json
import os
import statistics
import sys
import time
from pathlib import Path

ROOT = REPO_ROOT
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

import faiss  # noqa: E402
import numpy as np  # noqa: E402

from backend.pipeline import generator, sparse as sparse_module  # noqa: E402
from backend.pipeline.embedder import embed, model_key  # noqa: E402
from backend.pipeline.loader import load_pdf  # noqa: E402
from backend.pipeline.preprocessor import preprocess  # noqa: E402
from backend.pipeline.retriever import DENSE_WEIGHT, retrieve  # noqa: E402
from eval_common import judge  # noqa: E402

DECK_DIR = ROOT / "data" / "projects" / "AI"
EVAL_DIR = ROOT / "data" / "eval"
GROUND_TRUTH_PATH = EVAL_DIR / "slide_ground_truth.json"
OUT_PATH = EVAL_DIR / "own_material.json"
SHEET_PATH = EVAL_DIR / "own_material_rating_sheet.csv"

TOP_K = 3
STYLES = ["short", "explain"]

In [14]:
def covers(chunk, source_file, page) -> bool:
    if getattr(chunk, "source_file", None) != source_file:
        return False
    first = getattr(chunk, "page", None)
    if first is None:
        return False
    return first <= page <= (getattr(chunk, "page_end", None) or first)

In [15]:
def write_rating_sheet(results):
    """
    The answers in a random order, with no hint of how the rule scored them,
    for a person to mark. Columns: ok = 1 if the answer is right, 0 if not,
    blank if the question itself is unusable.
    """
    import csv
    import random

    rows = []
    for style in STYLES:
        for i, r in enumerate(results[style]["rows"]):
            rows.append({"id": f"{style[0]}{i:02d}", "style": style,
                         "question": r["question"],
                         "answer": " ".join(r["answer"].split()),
                         "reference_from_the_slide": r["expected"],
                         "source_file": r["source_file"], "slide": r["page"],
                         "ok": ""})
    random.Random(0).shuffle(rows)
    with open(SHEET_PATH, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)
    print(f"  -> {SHEET_PATH.relative_to(ROOT)}  ({len(rows)} answers to rate)")

In [16]:
def main():
    try:
        sys.stdout.reconfigure(encoding="utf-8", errors="replace")
    except Exception:
        pass

    truth = json.loads(GROUND_TRUTH_PATH.read_text(encoding="utf-8"))
    questions = truth["questions"]
    print(f"{len(questions)} questions over the student's own decks "
          f"({sum(1 for q in questions if q['source'] == 'picture')} of them "
          f"answerable only from text read out of a picture)")
    print(f"model {generator.MODEL_NAME}   embedder {model_key()}   "
          f"k={TOP_K}   abstain={generator.ABSTAIN}\n")

    paths = sorted(DECK_DIR.glob("*.pdf"))
    chunks = []
    for path in paths:
        chunks.extend(preprocess(load_pdf(str(path), figures="auto")))
    vectors = embed(chunks)
    index = faiss.IndexFlatL2(vectors.shape[1])
    index.add(np.ascontiguousarray(vectors, dtype=np.float32))
    keywords = sparse_module.build_index(chunks)
    print(f"{len(chunks)} chunks across {len(paths)} decks\n")

    passages, found = [], []
    for entry in questions:
        got = retrieve(entry["question"], index, chunks, k=TOP_K,
                       sparse=keywords, dense_weight=DENSE_WEIGHT)
        passages.append([str(c) for c in got])
        found.append(any(covers(c, entry["source_file"], entry["page"])
                         for c in got))

    results = {"retrieval_recall_at_3": round(sum(found) / len(found), 4)}
    print(f"retrieval found the source slide for {sum(found)}/{len(found)} "
          f"({sum(found) / len(found):.0%})\n")

    for style in STYLES:
        was = generator.ANSWER_STYLE
        generator.ANSWER_STYLE = style
        rows, seconds = [], []
        try:
            for entry, context, hit in zip(questions, passages, found):
                t0 = time.time()
                answer = generator.generate(entry["question"], context)
                seconds.append(time.time() - t0)
                correct, signals = judge(entry["answer"], answer)
                rows.append({
                    "question": entry["question"],
                    "expected": entry["answer"],
                    "source": entry["source"],
                    "source_file": entry["source_file"],
                    "page": entry["page"],
                    "retrieved_source_slide": hit,
                    "answer": answer,
                    "correct": bool(correct),
                    "signals": signals,
                })
        finally:
            generator.ANSWER_STYLE = was

        correct = sum(r["correct"] for r in rows)
        with_page = [r for r in rows if r["retrieved_source_slide"]]
        without = [r for r in rows if not r["retrieved_source_slide"]]
        picture = [r for r in rows if r["source"] == "picture"]
        text = [r for r in rows if r["source"] == "text_layer"]
        results[style] = {
            "correct": correct,
            "questions": len(rows),
            "accuracy": round(correct / len(rows), 4),
            "accuracy_when_slide_retrieved": round(
                sum(r["correct"] for r in with_page) / len(with_page), 4) if with_page else None,
            "accuracy_when_slide_missed": round(
                sum(r["correct"] for r in without) / len(without), 4) if without else None,
            "accuracy_text_layer": round(
                sum(r["correct"] for r in text) / len(text), 4) if text else None,
            "accuracy_from_pictures": round(
                sum(r["correct"] for r in picture) / len(picture), 4) if picture else None,
            "median_seconds": round(statistics.median(seconds), 2),
            "median_words": statistics.median(len(r["answer"].split()) for r in rows),
            "rows": rows,
        }
        r = results[style]
        print(f"  {style:<8} {correct:>2}/{len(rows)}  ({r['accuracy']:.0%})   "
              f"slide retrieved {r['accuracy_when_slide_retrieved']:.0%} / "
              f"missed {r['accuracy_when_slide_missed']:.0%}   "
              f"text {r['accuracy_text_layer']:.0%} / "
              f"picture {r['accuracy_from_pictures']:.0%}   "
              f"{r['median_seconds']:.1f}s, {r['median_words']:.0f} words")

    write_rating_sheet(results)

    OUT_PATH.write_text(json.dumps({
        "note": "The system end to end on the student's own lecture decks, "
                "the task it was built for. Questions come from "
                "slide_ground_truth.json, written by this system's own "
                "question model from single slides, so the reference answer "
                "is one this system gave with the slide in hand: this is a "
                "round-trip consistency measure, biased upward, not a measure "
                "of absolute correctness. Scored by the same judge rule as "
                "every other end-to-end number here.",
        "model": generator.MODEL_NAME,
        "embedder": model_key(),
        "abstain": generator.ABSTAIN,
        "k": TOP_K,
        "decks": len(paths),
        "chunks": len(chunks),
        "questions": len(questions),
        **results,
    }, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
    print(f"\n  -> {OUT_PATH.relative_to(ROOT)}")

In [17]:
if RUN:
    main()
else:
    print('RUN is False: showing the saved results below.')

RUN is False: showing the saved results below.


### Results

The reference answers were written by this system's own question model from single slides, so this is a round-trip consistency check, biased upward.

In [18]:
om = saved("own_material.json")
print(f"{om['questions']} questions, {om['chunks']} chunks, retrieval found the slide "
      f"{om['retrieval_recall_at_3']:.0%} of the time (k={om['k']})")
pd.DataFrame({s: {k: v for k, v in om[s].items() if k != "rows"}
              for s in ("short", "explain")}).T

51 questions, 228 chunks, retrieval found the slide 80% of the time (k=3)


,correct,questions,accuracy,accuracy_when_slide_retrieved,accuracy_when_slide_missed,accuracy_text_layer,accuracy_from_pictures,median_seconds,median_words
short,12.0,51.0,0.2353,0.2927,0.0,0.1842,0.3846,0.58,1.0
explain,15.0,51.0,0.2941,0.3415,0.1,0.2895,0.3077,2.69,32.0


## Reading slides as pictures

Measures what reading a page as a picture buys, and what it costs.

Slides are largely pictures: a diagram, a screenshot of code, a chart. When a
page's text layer is thin, loader._read_page_picture renders the page and
reads it, cheapest tool first — EasyOCR, and Qwen2-VL only when OCR comes back
with almost nothing and the picture covers most of the page
(FIGURE_VISION_IF_FEWER, FIGURE_VISION_MIN_AREA, at most FIGURE_VISION_PER_DOC
pages per document). The comparison of Qwen2-VL against EasyOCR+BLIP on DocVQA
says which reader is better; it says nothing about this cascade, which is what
this script measures.

Two questions, two parts:

```text
  1. What does it cost? A sample of candidate pages is read again with the
     cache bypassed, recording which tool answered, how many words came back
     and how long it took. Escalation to the vision model is rare by design
     and expensive when it happens, so both rates and seconds are reported.

  2. What does it recover? The picture questions in the slide ground truth
     were written only from text the text layer does not contain. Retrieval is
     run over an index built without reading pictures and over one built with
     it. The first is a floor: those answers are not in the index at all, and
     anything it retrieves is a coincidence of wording.
```

The text_layer questions are run through both indexes as a control: reading
pictures adds chunks to the corpus, and that could in principle push the
answers to ordinary questions down the ranking.

In [19]:
# The script's command-line options, as it would have read them.
sys.argv = ['notebook', '--sample', '40']

In [20]:
import json
import os
import random
import statistics
import sys
import time
from pathlib import Path

ROOT = REPO_ROOT
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

import faiss  # noqa: E402
import fitz  # noqa: E402
import numpy as np  # noqa: E402

from backend.pipeline import loader  # noqa: E402
from backend.pipeline import sparse as sparse_module  # noqa: E402
from backend.pipeline.embedder import embed, model_key  # noqa: E402
from backend.pipeline.preprocessor import preprocess  # noqa: E402
from backend.pipeline.retriever import DENSE_WEIGHT, retrieve  # noqa: E402

DECK_DIR = ROOT / "data" / "projects" / "AI"
EVAL_DIR = ROOT / "data" / "eval"
GROUND_TRUTH_PATH = EVAL_DIR / "slide_ground_truth.json"
OUT_PATH = EVAL_DIR / "figure_reading.json"

SEED = 5
K_VALUES = [1, 3, 5]
SAMPLE_PAGES = (int(sys.argv[sys.argv.index("--sample") + 1])
                if "--sample" in sys.argv else 40)

In [21]:
def cost_sample(paths, rng):
    """
    Read a sample of candidate pages with the cache bypassed, recording the
    tool that answered, the words recovered and the seconds taken.
    """
    candidates = []
    for path in paths:
        with fitz.open(str(path)) as doc:
            texts = [page.get_text() for page in doc]
            for i, text in enumerate(texts):
                if loader._needs_figure_reading(text, doc[i]):
                    candidates.append((str(path), i,
                                       loader._picture_share(doc[i])))
    rng.shuffle(candidates)
    chosen = candidates[:SAMPLE_PAGES]
    print(f"  {len(candidates)} candidate pages across {len(paths)} decks; "
          f"reading {len(chosen)} of them with the cache off\n")

    # Bypass the cache so the real cost is measured, and do not write the
    # result back. Writing it back is not harmless: the sample allows the
    # vision model on any page large enough, ignoring the per-document budget
    # the application applies, so a page could be cached with different text
    # from the one the application would have stored. An earlier version of
    # this script did write back and cost the corpus a third of its chunks.
    original_read = loader._cached_figure_text
    original_write = loader._cache_figure_text
    loader._cached_figure_text = lambda digest: None
    loader._cache_figure_text = lambda digest, text: None
    records = []
    try:
        for path, i, share in chosen:
            with fitz.open(path) as doc:
                may_describe = share >= loader.FIGURE_VISION_MIN_AREA
                t0 = time.time()
                text, method = loader._read_page_picture(
                    doc[i], i + 1, allow_vision=may_describe)
                records.append({
                    "file": Path(path).name,
                    "page": i + 1,
                    "picture_share": round(share, 3),
                    "vision_allowed": bool(may_describe),
                    "method": method or "nothing found",
                    "words": len(text.split()),
                    "seconds": round(time.time() - t0, 2),
                })
            r = records[-1]
            print(f"    {r['file'][:34]:<34} p.{r['page']:<3} "
                  f"{r['method']:<14} {r['words']:>4} words  {r['seconds']:>6.1f}s")
    finally:
        loader._cached_figure_text = original_read
        loader._cache_figure_text = original_write

    by_method = {}
    for r in records:
        m = by_method.setdefault(r["method"], {"pages": 0, "seconds": [], "words": []})
        m["pages"] += 1
        m["seconds"].append(r["seconds"])
        m["words"].append(r["words"])
    summary = {m: {"pages": v["pages"],
                   "share": round(v["pages"] / len(records), 3),
                   "median_seconds": round(statistics.median(v["seconds"]), 2),
                   "total_seconds": round(sum(v["seconds"]), 1),
                   "median_words": round(statistics.median(v["words"]), 1)}
               for m, v in sorted(by_method.items())}
    return {"candidate_pages": len(candidates), "sampled": len(records),
            "by_method": summary, "pages": records}

In [22]:
def build(pages_by_deck):
    chunks = []
    for pages in pages_by_deck:
        chunks.extend(preprocess(pages))
    embeddings = embed(chunks)
    index = faiss.IndexFlatL2(embeddings.shape[1])
    index.add(np.ascontiguousarray(embeddings, dtype=np.float32))
    return chunks, index, sparse_module.build_index(chunks)

In [23]:
def covers(chunk, source_file, page) -> bool:
    if getattr(chunk, "source_file", None) != source_file:
        return False
    first = getattr(chunk, "page", None)
    if first is None:
        return False
    return first <= page <= (getattr(chunk, "page_end", None) or first)

In [24]:
def evaluate(chunks, index, keywords, questions):
    ranks = []
    for entry in questions:
        got = retrieve(entry["question"], index, chunks, k=max(K_VALUES),
                       sparse=keywords, dense_weight=DENSE_WEIGHT)
        ranks.append(next((i for i, c in enumerate(got, 1)
                           if covers(c, entry["source_file"], entry["page"])), None))
    total = len(questions) or 1
    return {
        "recall": {f"recall_at_{k}": round(sum(1 for r in ranks if r and r <= k) / total, 3)
                   for k in K_VALUES},
        "mrr_top5": round(statistics.mean(1 / r if r else 0 for r in ranks), 4),
        "ranks": ranks,
    }

In [25]:
def main():
    # Slide text carries Unicode the Windows console cannot encode.
    try:
        sys.stdout.reconfigure(encoding="utf-8", errors="replace")
    except Exception:
        pass
    rng = random.Random(SEED)
    truth = json.loads(GROUND_TRUTH_PATH.read_text(encoding="utf-8"))
    picture_questions = [q for q in truth["questions"] if q["source"] == "picture"]
    text_questions = [q for q in truth["questions"] if q["source"] == "text_layer"]
    paths = sorted(p for p in DECK_DIR.glob("*.pdf"))

    print("WHAT IT COSTS")
    if SAMPLE_PAGES:
        cost = cost_sample(paths, rng)
    else:
        # --sample 0 keeps the cost measured by an earlier run and re-runs only
        # the retrieval half, which is cheap and does not touch any model.
        cost = json.loads(OUT_PATH.read_text(encoding="utf-8"))["cost"]
        print(f"  reusing the sample of {cost['sampled']} pages already recorded")
    print(f"\n  {json.dumps(cost['by_method'], indent=2)}")

    print("\nWHAT IT RECOVERS")
    print(f"  {len(picture_questions)} picture questions, "
          f"{len(text_questions)} text-layer questions as a control\n")

    indexes = {}
    for setting in ("off", "auto"):
        t0 = time.time()
        pages_by_deck = [load(p, setting) for p in paths]
        chunks, index, keywords = build(pages_by_deck)
        indexes[setting] = {
            "chunks": len(chunks),
            "pages_with_text": sum(len(p) for p in pages_by_deck),
            "index_seconds": round(time.time() - t0, 1),
            "picture_questions": evaluate(chunks, index, keywords, picture_questions),
            "text_layer_questions": evaluate(chunks, index, keywords, text_questions),
        }

    print(f"{'figures':<9} {'chunks':>7} {'R@1':>6} {'R@3':>6} {'R@5':>6} "
          f"{'MRR':>7}   questions")
    for setting, r in indexes.items():
        for label in ("picture_questions", "text_layer_questions"):
            rec = r[label]["recall"]
            print(f"{setting if label.startswith('picture') else '':<9} "
                  f"{r['chunks'] if label.startswith('picture') else '':>7} "
                  f"{rec['recall_at_1']:>6} {rec['recall_at_3']:>6} "
                  f"{rec['recall_at_5']:>6} {r[label]['mrr_top5']:>7}   {label}")

    OUT_PATH.write_text(json.dumps({
        "note": "Cost and benefit of reading PDF pages as pictures. The cost "
                "sample bypasses the figure cache so each page is read for "
                "real. The retrieval halves compare an index built with "
                "figures='off' against one built with figures='auto'; picture "
                "questions were written only from text the text layer does "
                "not contain, so the 'off' row is a floor, and the text-layer "
                "questions are a control for the extra chunks.",
        "embedder": model_key(),
        "dense_weight": DENSE_WEIGHT,
        "seed": SEED,
        "thresholds": {
            "vision_if_fewer_words": loader.FIGURE_VISION_IF_FEWER,
            "vision_min_picture_share": loader.FIGURE_VISION_MIN_AREA,
            "vision_pages_per_document": loader.FIGURE_VISION_PER_DOC,
            "render_dpi": loader.FIGURE_RENDER_DPI,
        },
        "cost": cost,
        "retrieval": indexes,
    }, indent=2) + "\n", encoding="utf-8")
    print(f"\n  -> {OUT_PATH.relative_to(ROOT)}")

In [26]:
def load(path, figures):
    return loader.load_pdf(str(path), figures=figures)

In [27]:
if RUN:
    main()
else:
    print('RUN is False: showing the saved results below.')

RUN is False: showing the saved results below.


### Results

In [28]:
fr = saved("figure_reading.json")
display(pd.DataFrame(fr["cost"]["by_method"]).T.rename_axis(
    f"{fr['cost']['sampled']} of {fr['cost']['candidate_pages']} thin pages"))
rows = []
for setting, r in fr["retrieval"].items():
    for group in ("picture_questions", "text_layer_questions"):
        rows.append({"pictures read": setting, "questions": group,
                     "chunks": r["chunks"], "index seconds": r["index_seconds"],
                     **r[group]["recall"], "MRR@5": r[group]["mrr_top5"]})
pd.DataFrame(rows).set_index(["pictures read", "questions"])

,pages,share,median_seconds,total_seconds,median_words
40 of 312 thin pages,,,,,
ocr,28.0,0.7,1.83,52.8,6.0
vision,12.0,0.3,267.00,3250.9,61.5


chunks  index seconds  recall_at_1  \
pictures read questions                                                  
off           picture_questions        201           10.8        0.000   
              text_layer_questions     201           10.8        0.711   
auto          picture_questions        228           69.4        0.385   
              text_layer_questions     228           69.4        0.711   

                                    recall_at_3  recall_at_5   MRR@5  
pictures read questions                                               
off           picture_questions           0.077        0.077  0.0385  
              text_layer_questions        0.895        0.895  0.7982  
auto          picture_questions           0.769        0.846  0.5577  
              text_layer_questions        0.816        0.842  0.7697